# Lesson 04 — Building makemore Part 3: Activations & Gradients, BatchNorm

- **GitHub issue:** [#4](https://github.com/majorgilles/karpathy_ml_course/issues/4)
- **Video:** https://youtu.be/P6sfmUTpUmc
- **Lesson guide:** [../README.md](../README.md)
- **Transcript:** [../transcript.md](../transcript.md)

Use this notebook for exploratory follow-along work. Move reusable code to `../src/`, lightweight checks to `../tests/`, and representative outputs to `../artifacts/`.

## 1. Load names and prepare the diagnostic workspace

This lesson starts from the same character-level name data as the previous MLP lesson so that later activation and gradient observations can be compared on the same prediction problem. `torch` supplies tensor operations, `torch.nn.functional` supplies neural-network helpers, and `matplotlib` is prepared for later plots. `%matplotlib inline` keeps those plots inside the notebook; the imported `float32` symbol is not used by the current cells.

The input is `data/raw/names.txt`, with a fallback path for running the kernel from this notebook's folder. `splitlines()` produces `words`, a Python list containing one complete name per element. The current output confirms the first eight names and a total of 32,033 names. Next, each character needs a stable numeric ID before it can appear in a model input.

In [1]:
import torch  # Tensor operations and model parameters.
import torch.nn.functional as F  # One-hot encoding and other neural-network helpers.
import matplotlib.pyplot as plt  # Visualizations used later in the lesson.
from sympy.codegen.ast import float32  # Current exploration import; not used by these cells yet.
%matplotlib inline

In [2]:
# Load every name; each line in names.txt becomes one training sequence.
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),  # Kernel launched from the repository root.
    Path("../../../data/raw/names.txt"),  # Kernel launched from this notebook folder.
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:8]  # Inspect a small sample before building numeric examples.

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# Confirm how many complete name sequences are available.
len(words)

32033

## 2. Map characters to model-friendly IDs

The model consumes integer token IDs rather than Python characters. The input is the complete `words` list: `''.join(words)` combines its text, `set(...)` keeps each character once, and `sorted(...)` gives the discovered letters a stable alphabetical order. Discovering the vocabulary from the data also avoids assuming in advance that every dataset contains exactly `a` through `z`.

`stoi` means string-to-index and maps each letter to IDs `1` through `26`; `itos` reverses that mapping for readable output. The boundary token `.` receives ID `0` and will represent both padding before a name and the end after its final letter. The output is therefore a 27-token vocabulary. Next, `BLOCK_SIZE` determines how many preceding token IDs form one input context.

In [4]:
# Collect the 26 lowercase letters once and sort them for reproducible IDs.
chars = sorted(list(set(''.join(words))))
# Reserve 0 for the boundary token, so letters begin at index 1.
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
# Invert the mapping: model indices back to printable characters.
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## 3. Choose the context-window length

A **hyperparameter** is a setting chosen outside gradient-based training. `BLOCK_SIZE = 3` means each input example contains the three token IDs immediately preceding the expected next character. For example, the beginning of `emma` creates the context-target pair `... → e`, where each `.` is boundary-token ID `0`.

This cell outputs only the integer configuration value; it does not build examples yet. The dataset builder uses it next to keep every context row at width three while the window slides through each name.

In [5]:
BLOCK_SIZE = 3

## 4. Build context-target rows and split complete names

`build_dataset` receives a list of complete names and uses `stoi` plus `BLOCK_SIZE` to produce aligned tensors. For each name, `context` starts as `[0, 0, 0]`. Every loop iteration appends that three-ID context to `X`, appends the one observed next-character ID to `Y`, and then slides the window forward. Iterating over `w + '.'` also creates the final example whose expected target is the end-of-name token.

Each row `X[i]` is one training example's input, while `Y[i]` is its single expected target. The vocabulary contains 27 possible next-token candidates, but this cell does not calculate their probabilities or a loss yet; a later forward pass will assign 27 scores and the probability indexed by `Y[i]` will contribute directly to that example's loss.

On a clean top-to-bottom run, `random.seed(42)` makes the name-level shuffle reproducible. The first 80% of complete names create `Xtr` and `Ytr`, the next 10% create `Xdev` and `Ydev`, and the final 10% create `Xte` and `Yte`. Splitting names before generating rows prevents transitions from one name leaking across splits. The current outputs are `182,625` training rows, `22,655` development rows, and `22,866` test rows, all with three context IDs and one target ID per row. Row percentages differ slightly from 80/10/10 because names have different lengths.

Only `Xtr` and `Ytr` should supply parameter updates. Development data is for comparing choices, and test data stays untouched until final evaluation. This is the notebook's current stopping point; the next lesson concept is to initialize the MLP before examining its activations and gradients.

In [6]:
# Turn one supplied name split into aligned context rows X and next-character targets Y.
def build_dataset(words: list) -> tuple[torch.Tensor, torch.Tensor]:
    X, Y = [], []
    for w in words:  # Keep every transition from this split together.
        context = [0] * BLOCK_SIZE  # Start with three boundary-token IDs: "...".

        for ch in w + ".":  # Include the final boundary token as a target.
            ix = stoi[ch]  # Integer ID of the character this context should predict.
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]  # Drop the oldest ID and append this target ID.

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)  # Inspect this split's context and target tensors.
    return X, Y

import random
random.seed(42)  # Reproduce the same name-level split on each rerun.
random.shuffle(words)  # Shuffle complete names before slicing; do not split individual transitions.
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# Build training, development, and test tensors from disjoint name groups.
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.int64 torch.Size([182625]) torch.int64
torch.Size([22655, 3]) torch.int64 torch.Size([22655]) torch.int64
torch.Size([22866, 3]) torch.int64 torch.Size([22866]) torch.int64
